# The Environmental Impact of Cyberattacks: A Study of System Resource Utilization and Energy Consumption

(French) L’impact environnemental des cyberattaques : Étude de la sollicitation des ressources système et de la consommation énergétique

## Imports

In [ ]:
from pathlib import Path
from typing import TypedDict

import chardet
import pandas as pd

## Constants

### Numbers

In [ ]:
BYTES_PER_KB: float = 1024.0

DECIMAL_PLACES_KB: int = 2

ENCODING_SAMPLE_SIZE_BYTES: int = 4096

### Paths

In [ ]:
RAW_DATA_DIR: Path = Path("data/raw")

### Strings

In [ ]:
CHARDET_ENCODING_KEY: str = "encoding"

DEFAULT_ENCODING: str = "utf-8"

ENCODING_ERROR_STRATEGY: str = "replace"

FILE_MODE_READ_BINARY: str = "rb"
FILE_MODE_READ_TEXT: str = "r"

NO_EXTENSION_LABEL: str = "no_ext"

## Definitions

In [ ]:
class File(TypedDict, total=False):
    path: Path
    relative_path: Path
    filename: str
    extension: str
    encoding: str
    size_kb: float
    content: str
    error: str

## Functions

### Standalone Functions

In [ ]:
def detect_encoding(
    file_path: Path, sample_size: int = ENCODING_SAMPLE_SIZE_BYTES
) -> str:
    """
    Detect the encoding of a file.

    Args:
        file_path (Path): The path to the file.
        sample_size (int): The number of bytes to sample from the file.

    Returns:
        str: The detected encoding.
    """

    try:
        with open(file_path, FILE_MODE_READ_BINARY) as f:
            sample_bytes: bytes = f.read(sample_size)

        return chardet.detect(sample_bytes).get(CHARDET_ENCODING_KEY, DEFAULT_ENCODING)

    except (OSError, UnicodeDecodeError) as e:
        print(f"[{e}] Failed to detect encoding for {file_path}")
        return DEFAULT_ENCODING

In [ ]:
def get_files(directory_path: Path) -> list[Path]:
    """
    List all files in a directory.

    Args:
        directory_path (Path): The directory to list files from.

    Returns:
        list[Path]: A list of all files in the directory.
    """

    if not directory_path.exists():
        raise ValueError(f"[ValueError] {directory_path} does not exist.")

    return sorted([f for f in directory_path.rglob("*") if f.is_file()])

In [ ]:
def read_file_content(file_path: Path, encoding: str) -> tuple[str, str | None]:
    """
    Read the content of a file.

    Args:
        file_path (Path): The path to the file.
        encoding (str): The encoding of the file.

    Returns:
        tuple[str, str | None]: A tuple containing the content of the file and an error message if any.
    """

    try:
        with open(
            file_path,
            FILE_MODE_READ_TEXT,
            encoding=encoding,
            errors=ENCODING_ERROR_STRATEGY,
        ) as f:
            return f.read(), None

    except (OSError, UnicodeDecodeError) as e:
        print(f"[{e}] Failed to read {file_path}")
        return "", str(e)

### Composite 1 Functions

In [ ]:
def create_file_entry(file_path: Path, base_dir_path: Path) -> File:
    """
    Create a file entry.

    Args:
        file_path (Path): The path to the file.
        base_dir_path (Path): The base directory path.

    Returns:
        File: A file entry.
    """

    encoding: str = detect_encoding(file_path)
    rel_path: Path = file_path.relative_to(base_dir_path)
    size_kb: float = round(file_path.stat().st_size / BYTES_PER_KB, DECIMAL_PLACES_KB)

    content: str
    error: str | None
    content, error = read_file_content(file_path, encoding)

    entry: File = {
        "path": file_path,
        "relative_path": rel_path,
        "filename": file_path.name,
        "extension": file_path.suffix.lower() or NO_EXTENSION_LABEL,
        "encoding": encoding,
        "size_kb": size_kb,
        "content": content,
    }

    if error:
        entry["error"] = error

    return entry

### Composite 2 Functions

In [ ]:
def build_dataset(directory_path: Path) -> list[File]:
    """
    Build a dataset of files.

    Args:
        directory_path (Path): The directory to build the dataset from.

    Returns:
        list[File]: A list of files.
    """

    files: list[Path] = get_files(directory_path)
    dataset: list[File] = []

    for file_path in files:
        try:
            dataset.append(create_file_entry(file_path, directory_path))
        except FileNotFoundError as fnfe:
            print(f"[{fnfe}] File was deleted during processing: {file_path}")
            continue
        except PermissionError as pe:
            print(f"[{pe}] Permission denied: {file_path}")
            continue

    return dataset

## Execution

### Exploration

In [ ]:
file_dataset: list[File] = build_dataset(RAW_DATA_DIR)

In [ ]:
df_dataset: pd.DataFrame = pd.DataFrame(file_dataset)
df_dataset